# PPE detector: train, evaluate and stress-test on a free GPU

Runs the whole Project 1 pipeline on **Google Colab** (Runtime → Change runtime type → T4 GPU) or **Kaggle** (Settings → Accelerator → GPU, Internet on, and *Add Input* → dataset `mugheesahmad/sh17-dataset-for-ppe-detection`).

Takes about 1.5–2 hours, most of it training. Results and weights end up in `ppe_results.zip`.

## 1. Get the code
Option A (recommended): push this repo to GitHub and clone it. Option B: upload `01-ppe-detector.zip` with the file panel and unzip.

In [ ]:
REPO_URL = "https://github.com/affan-h/industrial-safety-vision.git"  # set to "" if you uploaded a zip instead
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
ROOT = "/tmp" if ON_KAGGLE else "/content"  # on Kaggle keep bulky data out of /kaggle/working
if REPO_URL:
    !rm -rf {ROOT}/repo && git clone -q {REPO_URL} {ROOT}/repo
    %cd {ROOT}/repo/01-ppe-detector
else:
    !unzip -q -o /content/01-ppe-detector.zip -d /content
    %cd /content/01-ppe-detector

In [ ]:
!pip install -q ultralytics kagglehub
!python -m pytest -q tests

## 2. Get SH17 and build the 4-class dataset
On **Kaggle**, the attached dataset is used directly. On **Colab**, `kagglehub` downloads it (~14 GB, about 10 min). If it asks for credentials, create an API token on kaggle.com (Settings → API) and upload `kaggle.json` to `~/.kaggle/`.

In [ ]:
import glob
attached = glob.glob("/kaggle/input/**/train_files.txt", recursive=True)
if attached:
    raw = os.path.dirname(attached[0])
else:
    import kagglehub
    raw = kagglehub.dataset_download("mugheesahmad/sh17-dataset-for-ppe-detection")
print(raw)

In [ ]:
!python prepare_data.py --raw "{raw}" --out data/sh17_ppe

## 3. Fine-tune YOLO11n

In [ ]:
!python train.py --data data/sh17_ppe/sh17_ppe.yaml --epochs 50 --batch 32

## 4. Evaluate on the held-out test split

In [ ]:
!python evaluate.py --weights weights/ppe_yolo11n.pt --data data/sh17_ppe/sh17_ppe.yaml

## 5. Robustness: low light, haze, motion blur

In [ ]:
!python robustness.py --weights weights/ppe_yolo11n.pt --data data/sh17_ppe/sh17_ppe.yaml
from IPython.display import Image, display
display(Image("results/robustness.png")); display(Image("results/robustness_examples.png"))

## 6. Failure gallery

In [ ]:
!python failure_gallery.py --weights weights/ppe_yolo11n.pt --data data/sh17_ppe/sh17_ppe.yaml
import glob
for p in sorted(glob.glob("results/failures/worst_*.jpg"))[:4]:
    display(Image(p, width=700))

## 7. Package results

In [ ]:
!mkdir -p results/training && cp runs/ppe_yolo11n/results.png runs/ppe_yolo11n/confusion_matrix_normalized.png results/training/ 2>/dev/null; cp runs/test_eval/*.png results/training/ 2>/dev/null
!zip -q -r ppe_results.zip results weights
if ON_KAGGLE:
    !cp ppe_results.zip /kaggle/working/
    print("Done: download ppe_results.zip from the notebook's Output panel")
else:
    from google.colab import files
    files.download("ppe_results.zip")